# Notebook: Putting the "money" in Moneyball
In this notebook we will:
* look at the new `batting` and `salaries` dataset
* understand how to merge different datasets
* use machine learning to predict salaries
* identify undervalued players

# Milestone 1: Introducing the `batting` and `salaries` datasets

In [ ]:
#@title Run this to download data and prepare our environment!
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ------------------ BATTING ------------------
batting = pd.read_csv("https://drive.google.com/uc?id=17IM9kxyscGEpu-4Ov5p7MpCKz9UVHLla")

# Rename columns to be more descriptive
abbrev_map = {
    "R" : "Runs",
    "AB" : "AtBats",

    "H"  : "Hits",
    "2B" : "Doubles",
    "3B" : "Triples",
    "HR" : "HomeRuns",
    "RBI": "RunsBattedIn",

    "BB"  : "Walks",
    "IBB" : "IntentionalWalks",
    "HBP" : "HitsByPitch",
    "SF"  : "SacrificeFlies",
    "SH"  : "SacrificeHits",

    "SB"   : "StolenBases",
    "CS"   : "CaughtStealing",
    "SO"   : "Strikeouts",
    "GIDP" : "GroundedIntoDoublePlays",

    "W" : "Wins",
    "L" : "Losses",
    "G" : "Games",
}

batting = batting.rename(columns=abbrev_map)
batting = batting.loc[batting["AtBats"] > 130] # remove rookies' rows

# Take data from multiple stints and put it in one row
batting = batting.groupby(['playerID', 'yearID']).sum(numeric_only=True)
batting = batting.sort_values(['playerID', 'yearID'])
batting = batting.reset_index()
batting = batting.drop(columns='stint')

# Add in number of seasons played
seasonsPlayed = batting.groupby('playerID')['yearID'].cumcount()
batting.insert(batting.columns.get_loc('yearID') + 1, 'SeasonsPlayed', seasonsPlayed)


# Add in features from NB1
batting["BattingAverage"] = batting.eval("Hits / AtBats")
batting["OnBasePercentage"] = batting.eval("(Hits + Walks + HitsByPitch) / (AtBats + Walks + HitsByPitch + SacrificeFlies)")
batting["SluggingPercentage"] = batting.eval("(Hits + Doubles + 2*Triples + 3*HomeRuns) / AtBats")


# ------------------ SALARIES ------------------
# load in salaries data
salaries = pd.read_csv("https://drive.google.com/uc?id=16wlS8fmyZ43sATfeSZ8eUAqR5543pg_G")
salaries = salaries.loc[salaries['salary'] > 0]
salaries = salaries.drop(columns="lgID")

# Remove players with multiple stints
salaries = salaries[~salaries.duplicated(subset=['playerID', 'yearID'], keep=False)]

# For people's names
people = pd.read_csv("https://drive.google.com/uc?id=1QkDGU9p0L1JqiubZwQl_ZqVbGltbWwKF")

## Exploring the Batting Dataset

We'll start by focusing on individual player performance. The `batting` dataset offers a wealth of information about players' batting statistics. *Note that this is different from the `teams` dataset we were using in the previous notebook, since this includes statistics for each player each year they played!*



In [ ]:
pd.set_option('display.max_columns', None)
batting.head(9)

## Adding in Seniority


Adding features that will tell us if our baseball players are arbitration eligible or free agents. Here are some descriptions:

- **Arbitration Eligible**: In baseball, players become eligible for arbitration after three to six years of service time. During arbitration, players negotiate contracts with their teams for the upcoming season, or an arbitrator decides on a salary if an agreement isn't reached.

- **Free Agent**: In baseball, a free agent is a player not under contract with any team, allowing them to sign with any team they choose. Free agency typically occurs when a player's contract expires, or after they've completed six years of service time in the Major Leagues, granting them the freedom to negotiate new contracts with various teams.


Why might adding these columns in be important for our later salary prediction?

### ✍ Adding in the Seniority Columns


In [ ]:
batting["isArbitrationEligible"] = (3 <= batting["SeasonsPlayed"]) & (batting["SeasonsPlayed"] < 6)
batting["isFreeAgent"] = 6 <= batting["SeasonsPlayed"]
batting["isRookie"] = (batting["SeasonsPlayed"] < 3)
batting.head(9)

# Examining the Salaries Dataset

The `salaries` dataset provides detailed information about the salaries of baseball players.


In [ ]:
salaries.head()

### ✍ Exploring Summary Statistics

 The `.describe()` method will provide key statistical measures for each column in the dataset, such as mean, standard deviation, minimum, maximum, and quartile values.


In [ ]:
salaries.describe()

### ✍ Plotting the Salary Data

A histogram is a type of plot that allows to plot how frequently values occur in a dataset. The $x$-axis will represent the data values of interest, while the $y$-axis will represent the counts!


In [ ]:
sns.histplot(x="salary", data=salaries)

### ✍ Transforming Salary Data with Logarithm

- Create a new column in the `salaries` dataset named `"LogSalary"`.
- This column should contain the logarithm of each player's salary.
- Once the new column is added, plot the distribution of the transformed salaries.


In [ ]:
salaries["LogSalary"] = np.log10(salaries["salary"])
sns.histplot(x="LogSalary", data=salaries)

### ✍ Examining Our Year Feature

We'll make a new column `yearID_salary` in the `batting` dataset so that we can link a player's batting data in one year to his salary the following year.


In [ ]:
batting['yearID_salary'] = batting['yearID'] + 1
batting.head(1)

In [ ]:
salaries = salaries.rename(columns={'yearID': 'yearID_salary', 'teamID': 'teamID_salary'})
salaries.head(1)

## Merging Datasets: Linking Player Data with Salaries

By merging these datasets, we'll be able to conduct more holistic analyses, such as assessing the impact of specific performance metrics on salary and exploring trends in player compensation over time.



### ✍ Merging the Datasets


In [ ]:
merged_data = pd.merge(batting, salaries, on=['yearID_salary','playerID'])
merged_data.head()

# Milestone 2: Salary Prediction

- Achieve a high degree of accuracy in salary predictions.
- Understand the factors that most significantly influence player salaries.
- Explore the relationship between player performance, experience, and compensation.



### Choosing the columns to focus on


In [ ]:
#@title Run this to choose the columns
#@markdown NOTE: as you toggle the columns below, your `X` dataset will automatically update! Keep note of this if you change things after trying out your models below.
from ipywidgets import widgets
from sklearn.preprocessing import MinMaxScaler

# Create checkboxes for each column
checkboxes = {col:
                widgets.Checkbox(description=col, value=(col not in ['salary', 'LogSalary', 'playerID', 'teamID_salary']))
                for col in merged_data.columns}

# Initial selection
bool_list = [checkboxes[col].value for col in merged_data.columns]
X = merged_data.loc[:, bool_list]
X_scaled = MinMaxScaler().fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)
y = merged_data["LogSalary"]

# Function to update selected_columns based on checkbox values
def update_selected_columns(change):
  global X, bool_list
  bool_list = [checkboxes[col].value for col in merged_data.columns]
  X = merged_data.loc[:, bool_list]
  X_scaled = MinMaxScaler().fit_transform(X)
  X = pd.DataFrame(X_scaled, columns=X.columns)

# Attach the callback to checkbox changes
for checkbox in checkboxes.values():
    checkbox.observe(update_selected_columns, 'value')

# Display checkboxes in a grid
widgets.GridBox(list(checkboxes.values()), layout=widgets.Layout(grid_template_columns="repeat(3, 200px)"))

Last step is to split the data as always! Why do we need to do this?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

### ✍ Initializing and fitting the model


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)


### ✍ Evaluating the model


In [ ]:
linear_model.score(X_test, y_test)

In [ ]:
predictions = linear_model.predict(X_test)
mean_squared_error(y_test, predictions)

In [ ]:
#@title Run this to see the linear regression model importances!

# Get absolute values of the coefficients of the model
linear_model_importances = np.abs(linear_model.coef_)

# Normalize feature importances
linear_model_importances /= linear_model_importances.sum()

# Plot feature importances
plt.bar(X.columns, linear_model_importances, label='Linear Regression')
plt.title('Feature Importances According to Linear Regression')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.xticks(rotation=90)
plt.show()

## ✍ Decision Tree!


In [ ]:
dt_model = DecisionTreeRegressor(max_depth=3)
dt_model.fit(X_train, y_train)
predictions = dt_model.predict(X_test)
print(mean_squared_error(y_test, predictions))
print(dt_model.score(X_test, y_test))



In [ ]:
#@title Run this to see the decision tree model importances!

# Get absolute values of the coefficients of the model
dt_importances = dt_model.feature_importances_

# Plot feature importances
plt.bar(X.columns, dt_importances)
plt.title('Feature Importances According to Decision Tree')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.xticks(rotation=90)
plt.show()

## Exploring Other Models

- `RandomForestRegressor`: basically a "forest" of decision trees that are each fit on different samples of the data. We determine the prediction by feeding the input to all the trees and taking the average of their predictions.
- `MLPRegressor`: a fully connected neural network. Remember to specify the hidden layer structure using the `hidden_layer_sizes` parameter!
-

In [ ]:
models = [RandomForestRegressor(max_depth = 6), MLPRegressor(hidden_layer_sizes=(64,32)),]

for model in models:
  model.fit(X_train, y_train)
  predictions = model.predict(X_test)
  print(model.score(X_test,y_test))
  print(mean_squared_error(y_test, predictions))

# Milestone 3: Determining the most undervalued players

To identify the most undervalued players, we will take a look at the predicted `LogSalary` and compare it to the actual value. In the code cell below, make a new column called `LogSalaryDifference` that is the result of subtracting the true `LogSalary` from the prediction of your chosen model.

In [ ]:
predictions = linear_model.predict(X)
merged_data['LogSalaryDifference'] = merged_data['LogSalary'] - predictions

Now what does this difference really mean? Let's dive back into the math. Here's what we have above:

$$ \text{LogSalaryDifference} = \log_{10}(\text{salary}_{\text{pred}}) - \log_{10}(\text{salary}_{\text{true}}) $$

A nifty rule of logarithms is that this is equivalent to

$$ \text{LogSalaryDifference} = \log_{10} \left( \frac{ \text{salary}_{\text{pred}} }{ \text{salary}_{\text{true}} } \right)$$

Another nifty rule of the logarithm we're using is that

\begin{align}
10^{\log_{10} x} = x
\end{align}

Putting all of this together, we have the following:

\begin{align}
10^\text{LogSalaryDifference} &= 10^{\log_{10} \left( \frac{ \text{salary}_{\text{pred}} }{ \text{salary}_{\text{true}} } \right)} \\
&= \frac{ \text{salary}_{\text{pred}} }{ \text{salary}_{\text{true}} }
\end{align}

meaning we can get the ratio of the predicted salary to the true salary by taking the exponent of our `LogSalaryDifference` column!



In [ ]:
merged_data['SalaryRatio'] = np.power(10, merged_data['LogSalaryDifference'])

Run the cell below to sort the data by this ratio! What do values above 1 mean? What about below 1? Who are the most under- and overvalued players based on our model?

In [ ]:
merged_data.sort_values('SalaryRatio')